## WP013 stage 2 — continuity covariate in the model: walk-forward CV

**Heavy compute — run this yourself.** Same pattern as WP005/009/011: an 18-window screen, then the full 35 only if the screen shows something.

Arms (all evaluated on the same held-out EPL matches):
- `baseline` — WP001, seeded (not re-run).
- `lineup_loose_combo` — WP009's best config, seeded (not re-run).
- **`continuity`** — WP001 defaults plus `use_continuity` (opponent's defence continuity, standardised, one shared `beta_continuity ~ Normal(0, 0.1)`).
- **`continuity_lineup_loose_combo`** — WP009's `lineup_loose_combo` plus `use_continuity`.

**Pre-declared primary comparison:** paired RPS, `continuity − baseline`, 95% bootstrap CI entirely below zero. Secondary: `continuity_lineup_loose_combo − lineup_loose_combo`. Also reported: `beta_continuity` per window (the hypothesis says negative).

**Caveat that cannot be removed:** the idea came from a residual check on these same 401 held-out matches (WP013 stage 1: slope −0.077 goals/SD, CI [−0.168, +0.013]), so a hit here is not an independent confirmation. Detection limit for a paired RPS difference on ~361 matches is about ±0.0007–0.001 (WP011).

In [ ]:
import pickle, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk
from football_model.types.model_data import ModelConfig

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP009 = REPO / 'work_products' / 'wp009_lineup_xg_validation'
WP013 = REPO / 'work_products' / 'wp013_lineup_continuity'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'
sys.path.insert(0, str(REPO / 'scripts'))
from run_cv_window import run_windows_concurrent  # noqa: E402

DATA_PATH = WP013 / 'cv_shared_data.pkl'     # WP009's shared data + 'continuity_table'
with open(DATA_PATH, 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
assert 'continuity_table' in shared and 'lineup_dev_table' in shared
odds = pd.read_pickle(WP003 / 'odds_raw.pkl').dropna(subset=['Date', 'FTR']).reset_index(drop=True)
print(len(windows), 'windows;', len(df_cv), 'EPL rows;', len(shared['continuity_table']), 'continuity rows')

LOOSE = dict(init_scale=0.30, home_adv_sd=0.06, sigma_att=0.020, sigma_def=0.020)   # WP005's loose_combo
ARMS = {
    'baseline':                      None,   # seeded from WP001
    'lineup_loose_combo':            None,   # seeded from WP009 (full 35-window checkpoint)
    'continuity':                    {'use_continuity': True},
    'continuity_lineup_loose_combo': {'use_continuity': True, 'use_lineup_xg': True, **LOOSE},
}
BASE = dict(clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True)
for name, ov in ARMS.items():
    if ov is not None:
        ModelConfig(**BASE, **ov)
RUN_ARMS = [n for n, ov in ARMS.items() if ov is not None]
print('arms to run:', RUN_ARMS)

MAX_WORKERS = 3     # conservative; 12 chain threads on a 12-core laptop throttles (see WP012 discussion)
WINDOW_TIMEOUT = 1800

def load_ckpt(p):
    p = Path(p)
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

def seed_from(src, dest_name, windows_subset):
    dest = WP013 / f'cv_checkpoint_{dest_name}.pkl'
    if dest.exists():
        return
    s = load_ckpt(src)
    filt = {'results': [r for r in s['results'] if r['window'] in windows_subset],
            'cv_match_predictions': [m for m in s['cv_match_predictions'] if m['window'] in windows_subset]}
    pickle.dump(filt, open(dest, 'wb'))
    print(f'seeded {dest_name} from {Path(src).name}:', len(filt['results']), 'windows')

## Phase 2 — screening (18 windows)

In [ ]:
SCREEN_WINDOWS = list(range(1, len(windows) + 1, 2))
seed_from(WP001 / 'cv_checkpoint.pkl', 'baseline', SCREEN_WINDOWS)
seed_from(WP009 / 'cv_checkpoint_full_lineup_loose_combo.pkl', 'lineup_loose_combo', SCREEN_WINDOWS)

# Time ONE window first: continuity adds one scalar and two data arrays, so it should cost about the same as WP009's runs.
t0 = time.time()
run_windows_concurrent(SCRIPT, DATA_PATH, WP013 / 'cv_checkpoint_continuity.pkl', [SCREEN_WINDOWS[0]],
                       config_overrides=ARMS['continuity'], max_workers=1, timeout=WINDOW_TIMEOUT)
print(f'one window: {time.time() - t0:.1f}s')

In [ ]:
t0 = time.time()
for name in RUN_ARMS:
    run_windows_concurrent(SCRIPT, DATA_PATH, WP013 / f'cv_checkpoint_{name}.pkl', SCREEN_WINDOWS,
                           config_overrides=ARMS[name], max_workers=MAX_WORKERS, timeout=WINDOW_TIMEOUT)
print(f'\nscreening wall time: {(time.time() - t0) / 60:.1f} min')
for name in ARMS:
    print(f'  {name}: {len(load_ckpt(WP013 / f"cv_checkpoint_{name}.pkl")["results"])}/{len(SCREEN_WINDOWS)}')

### Analysis — used for both the screen and the full run

Joins each arm's held-out predictions to Pinnacle's closing odds (the join raises if any score disagrees), then reports each arm's gap to Pinnacle and the paired RPS differences declared above. `beta_continuity` per window is summarised too.

In [ ]:
need = [f'PSC{o}' for o in mk.OUTCOMES]

def frame(ckpt):
    return mk.join_odds(mk.model_fixtures(df_cv, windows, ckpt), odds, required_cols=need)

def analyse(prefix):
    ckpts = {n: load_ckpt(WP013 / f'cv_checkpoint_{prefix}{n}.pkl') for n in ARMS}
    missing = [n for n, c in ckpts.items() if not c['results']]
    if missing:
        print('not run yet:', missing); return
    J = {n: frame(c) for n, c in ckpts.items()}
    keys = J['baseline'][['date', 'home_fd', 'away_fd']]
    for n, j in J.items():
        assert j[['date', 'home_fd', 'away_fd']].equals(keys), f'{n} covers different matches than baseline'
    y = mk.outcome_onehot(J['baseline']['FTR'])
    pin = mk.devig(mk.odds_matrix(J['baseline'], 'PSC'))
    R = {n: mk.rps(j[['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy(), y) for n, j in J.items()}
    rp = mk.rps(pin, y)
    print(f'n = {len(y)} matches with Pinnacle closing odds; Pinnacle RPS {rp.mean():.4f}\n')
    rows = []
    for n in ARMS:
        m, lo, hi = mk.bootstrap_ci(R[n] - rp, 5000)
        rows.append({'arm': n, 'rps': R[n].mean(), 'gap_to_pinnacle': m, 'lo': lo, 'hi': hi})
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print()
    rows = []
    for label, a, b in [('PRIMARY   continuity − baseline', 'continuity', 'baseline'),
                        ('secondary continuity_lineup_loose_combo − lineup_loose_combo', 'continuity_lineup_loose_combo', 'lineup_loose_combo'),
                        ('context   lineup_loose_combo − baseline', 'lineup_loose_combo', 'baseline')]:
        m, lo, hi = mk.bootstrap_ci(R[a] - R[b], 5000)
        rows.append({'paired RPS difference (negative = first is better)': label, 'mean': m, 'lo': lo, 'hi': hi})
    print(pd.DataFrame(rows).round(5).to_string(index=False))
    print()
    for n in ('continuity', 'continuity_lineup_loose_combo'):
        b = np.array([r['beta_continuity'] for r in ckpts[n]['results']], float)
        print(f'beta_continuity, {n}: mean {b.mean():+.4f}  sd {b.std():.4f}  range [{b.min():+.4f}, {b.max():+.4f}]  negative in {(b < 0).sum()}/{len(b)} windows')

analyse('')

## Phase 3 — full 35 windows

Only worth running if the screen shows something (a primary CI clearly below zero, or a `beta_continuity` that is consistently negative and well away from zero). Otherwise stop here, per the same discipline as WP005–011.

In [ ]:
FULL_WINDOWS = list(range(1, len(windows) + 1))
seed_from(WP001 / 'cv_checkpoint.pkl', 'full_baseline', FULL_WINDOWS)
seed_from(WP009 / 'cv_checkpoint_full_lineup_loose_combo.pkl', 'full_lineup_loose_combo', FULL_WINDOWS)

t0 = time.time()
for name in RUN_ARMS:
    run_windows_concurrent(SCRIPT, DATA_PATH, WP013 / f'cv_checkpoint_full_{name}.pkl', FULL_WINDOWS,
                           config_overrides=ARMS[name], max_workers=MAX_WORKERS, timeout=WINDOW_TIMEOUT)
print(f'\nfull CV wall time: {(time.time() - t0) / 60:.1f} min')
analyse('full_')